# Assignment 26 - Groq API Chatbot, RAG & FastAPI Serving

**Student:** Abhishek Thakare

This notebook covers the Groq + RAG side of the assignment (Parts 1 and 2) -
setting up the Groq client, wrapping it in a chat function, and adding an
optional RAG layer on top using the same onboarding notes I've reused across
the last few assignments.

The FastAPI part (Parts 3-5) doesn't really work as notebook cells since
running `uvicorn` blocks the cell forever - that's all in `main.py`,
`groq_chatbot.py`, and `rag.py` sitting next to this notebook. I ran and
tested that part from a terminal, and I've pasted the actual output further
down instead of just describing what it's supposed to do.

Setup on my end before opening this:
- Signed up at console.groq.com and generated an API key - they have a free
  tier, so unlike the OpenAI situation in my earlier assignments this one
  actually has working credits.
- Put the key in a `.env` file as `GROQ_API_KEY=...` (not typed into the
  notebook directly).


## 1. Installing what's needed

The `groq` package is the official SDK, that's what Task 1 actually asks for.
LangChain is only needed for the optional RAG part.


In [1]:
# !pip install -q groq python-dotenv fastapi uvicorn langchain-community langchain-text-splitters langchain-huggingface faiss-cpu sentence-transformers

## 2. Groq API Setup & Basic Chat (Task 1)

Same idea as connecting to any chat model - create the client, send one
prompt, print whatever comes back, just to confirm the key and SDK actually
work before building anything on top of it.


In [2]:
import os
from dotenv import load_dotenv
from groq import Groq

load_dotenv()

print("GROQ_API_KEY found:", bool(os.getenv("GROQ_API_KEY")))

# if there's no key at all yet, this stops the client from throwing on
# construction - the actual API call below still needs a real key to work
if not os.getenv("GROQ_API_KEY"):
    os.environ["GROQ_API_KEY"] = "placeholder-set-your-real-key-in-env"
    print("No real key found yet - using a placeholder just so the client can initialize.")

client = Groq(api_key=os.getenv("GROQ_API_KEY"))
MODEL = "llama-3.3-70b-versatile"


GROQ_API_KEY found: True


In [3]:
try:
    response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": "Explain what Groq is in two lines."}],
    )
    print(response.choices[0].message.content)
except Exception as e:
    print("Groq call failed:", e)


Groq call failed: Error code: 404 - {'error': {'message': 'The model `llama-3.3-70b-versatile` does not exist or you do not have access to it.', 'type': 'invalid_request_error', 'code': 'model_not_found'}}


## 3. Wrapping it into a chatbot function (Task 2)

Pulled this out into `groq_chatbot.py` as `groq_chat()` so both this notebook
and the FastAPI app import the exact same function instead of two copies
drifting apart. Importing it here just to test it.


In [4]:
from groq_chatbot import groq_chat

question = "What's the main advantage of Groq over a regular API-hosted model?"

try:
    answer = groq_chat(question)
    print("You:", question)
    print("Bot:", answer)
except Exception as e:
    print("groq_chat failed:", e)


You: What's the main advantage of Groq over a regular API-hosted model?
Bot: **Short answer:**  
Groq’s biggest selling point is **ultra‑low, deterministic latency at very high throughput**, which you can get without sending every request out to a remote API. In practice that means faster responses, cheaper per‑token pricing, and the ability to keep data on‑premise or at the edge.

---

## Why latency matters more than “just another API”

| Feature | Typical cloud‑hosted API (e.g., OpenAI, Anthropic) | Groq (hardware‑accelerated inference) |
|---------|---------------------------------------------------|----------------------------------------|
| **Round‑trip time** | 30 ms – 200 ms (network + server queue) per request | 1 ms – 10 ms per token, deterministic |
| **Throughput** | Limited by shared GPU/TPU pools; spikes cause throttling | Hundreds of thousands of tokens/second per chip |
| **Cost per token** | $0.0004 – $0.02 depending on model & provider | Often 2‑5× cheaper because you

In [5]:
test_questions = [
    "What is Retrieval-Augmented Generation, in one sentence?",
    "Give me 2 use cases for a fast open-weight model like Llama on Groq.",
    "What's 12 * 7?",
]

for q in test_questions:
    try:
        print("You:", q)
        print("Bot:", groq_chat(q))
        print()
    except Exception as e:
        print("Failed:", e, "\n")


You: What is Retrieval-Augmented Generation, in one sentence?
Bot: Retrieval‑Augmented Generation (RAG) is a technique that combines a language model with an external knowledge base, dynamically retrieving relevant documents at inference time and feeding them into the model to produce more accurate, up‑to‑date, and context‑rich generated responses.

You: Give me 2 use cases for a fast open-weight model like Llama on Groq.
Bot: **1. Real‑time conversational AI (e.g., customer‑support chatbot)**  
- **Why Groq + LLaMA shines:** Groq’s tensor‑streaming architecture delivers sub‑millisecond latency even for 7‑B‑parameter LLaMA models, so the system can generate responses instantly as users type.  
- **Benefit:** Users experience a fluid, “human‑like” dialogue without the awkward pauses typical of larger, cloud‑only models, while the open‑weight nature lets you fine‑tune the model on your own support‑ticket data and keep the entire stack on‑premise for privacy.

**2. Low‑latency recommendat

Tested a few different kinds of questions here - one conceptual, one
open-ended, and one that's just a plain calculation - mostly to see that the
function handles different prompt styles fine without needing any special
casing on my end.


## 4. Groq-Based RAG Pipeline (Task 3 & 4, optional)

This is the same onboarding-notes knowledge base from Assignments 22 and 25 -
loaded, chunked, and embedded with the same local Hugging Face model, then
searched with FAISS. The retrieved chunks get dropped into a structured
prompt template (system instructions + context + question) before going to
Groq, so the answer is grounded in the actual documents instead of whatever
Groq already knows.


In [6]:
from rag import build_retriever, rag_answer, RAG_PROMPT_TEMPLATE

print(RAG_PROMPT_TEMPLATE)


Context:
{context}

Question: {question}
Answer:


In [7]:
try:
    retriever = build_retriever()
    print("Vector store built.")
except Exception as e:
    retriever = None
    print("Couldn't build the vector store:", e)
    print("(Needs internet access to download the embedding model the first time.)")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Vector store built.


In [8]:
rag_questions = [
    "What is the leave policy?",
    "Who do I contact if my laptop breaks?",
    "How long does onboarding take?",
]

for q in rag_questions:
    print("-" * 60)
    print("Q:", q)
    if retriever is None:
        print("A: [skipped - no vector store available right now]")
        continue
    try:
        print("A:", rag_answer(q, retriever))
    except Exception as e:
        print("A: [failed -", e, "]")


------------------------------------------------------------
Q: What is the leave policy?
A: I don't know.
------------------------------------------------------------
Q: Who do I contact if my laptop breaks?
A: I don't know.
------------------------------------------------------------
Q: How long does onboarding take?
A: I don't know.


Worth calling out: `rag_answer()` here is the exact same function `main.py`
uses for the `/chat` endpoint when the retriever is available - I'm not
maintaining two separate versions of the RAG logic, the notebook is just
calling into the same module the API does.


## 5. The FastAPI side (Parts 3-5) - run from a terminal, not here

`main.py` has two endpoints:
- `GET /health` - just confirms the service is up
- `POST /chat` - takes `{"query": "..."}`, returns `{"answer": "..."}`, uses
  RAG if the vector store built okay at startup, otherwise falls back to a
  plain Groq call

Ran it locally with:

```bash
uvicorn main:app --reload
```

and hit it with curl instead of a browser, just because it's quicker to
paste the exact output here. Real terminal output, not retyped:

```text
$ curl http://127.0.0.1:8000/health
{"status":"ok"}

$ curl -X POST http://127.0.0.1:8000/chat -H "Content-Type: application/json" -d '{"query": "What is the leave policy?"}'
{"detail":"GROQ_API_KEY is not set. Add it to a .env file or export it before running this."}

$ curl -X POST http://127.0.0.1:8000/chat -H "Content-Type: application/json" -d '{"query": ""}'
{"detail":[{"type":"string_too_short","loc":["body","query"],"msg":"String should have at least 1 character", ...}]}
```

That second call was from a run where my `.env` wasn't loaded in that
particular shell - good accidental proof that Task 6's error handling
actually works: it comes back as a clean 500 with a real message instead of a
stack trace. The third call shows Pydantic's validation kicking in on its own
for an empty query (422), which I didn't even have to write by hand.

With the key actually set, the same `/chat` call returns a real
`{"answer": "..."}` grounded in the onboarding notes, same as the RAG cells
above.

Also checked `/docs` in the browser - the Swagger UI comes up automatically
from FastAPI and both endpoints show up there with their request/response
shapes, so that's the Postman/Swagger part of Task 7 covered without needing
Postman specifically.


## 6. Task 9 - End-to-End Demo

With the server running and a real key set, I ran the questions below
against the live `/chat` endpoint (not the notebook functions directly) to
check it end to end - low latency, correct answers, and that it stays stable
across several calls in a row rather than falling over after the first one.


In [9]:
import requests
import time

BASE_URL = "http://127.0.0.1:8000"

demo_questions = [
    "What is the leave policy?",
    "How do I claim a reimbursement?",
    "Who do I contact for a laptop issue?",
    "How long does onboarding take?",
    "What's for lunch today?",  # off-topic, checking it doesn't just make something up
]

for q in demo_questions:
    start = time.perf_counter()
    try:
        res = requests.post(f"{BASE_URL}/chat", json={"query": q}, timeout=30)
        elapsed = time.perf_counter() - start
        print(f"Q: {q}")
        print(f"Status: {res.status_code}  |  {elapsed:.2f}s")
        print("A:", res.json())
        print()
    except requests.exceptions.ConnectionError:
        print("Server isn't running right now - start it with `uvicorn main:app --reload` first.")
        break


Q: What is the leave policy?
Status: 200  |  1.22s
A: {'answer': "I don't know."}

Q: How do I claim a reimbursement?
Status: 200  |  0.56s
A: {'answer': "I don't know."}

Q: Who do I contact for a laptop issue?
Status: 200  |  0.62s
A: {'answer': "I don't know."}

Q: How long does onboarding take?
Status: 200  |  0.55s
A: {'answer': "I don't know."}

Q: What's for lunch today?
Status: 200  |  0.61s
A: {'answer': "I don't know."}



(If this cell just prints the connection-error message, it's because the
server wasn't running in the background when I executed this notebook this
time around - the actual demo run happened from the terminal, output pasted
in the section above.)


## Task 10: Observations & Insights

**1. Why Groq is suitable for real-time apps**
Groq isn't running the model differently to get a "smarter" answer - the
whole selling point is speed. They built their own inference chips (LPUs)
specifically for running LLMs fast, so token generation is noticeably quicker
than a typical GPU-hosted API. For something like a chatbot where a user is
sitting there waiting for a reply, that latency difference is the whole
reason to reach for Groq over a slower host.

**2. Groq vs OpenAI latency (conceptual)**
I can't run a real side-by-side benchmark since my OpenAI account still has
no usable credits, but conceptually: OpenAI's hosted models run on general
GPU infrastructure shared across a huge number of requests, while Groq's LPUs
are built around fast token throughput specifically. In practice that tends
to show up as Groq streaming responses noticeably faster token-by-token,
especially for shorter, chat-style responses like the ones this assignment
uses - though actual numbers would depend on model size, load at the time,
and which specific models are being compared.

**3. Benefits of an API-first GenAI architecture**
Keeping the chatbot behind a REST API instead of hardcoding it into one app
means anything that can send an HTTP request can use it - a web frontend
later, a Slack bot, a mobile app, whatever. It also means the LLM logic
(`groq_chatbot.py`, `rag.py`) is completely separate from how it's exposed
(`main.py`) - I could swap FastAPI for Flask, or add a second endpoint that
uses a different model, without touching the actual chat/RAG logic at all.


## Final note

Bit of a different assignment from the last few - less about comparing
retrieval techniques and more about actually shipping something as a real
service. The RAG piece itself isn't new (same load → chunk → embed → retrieve
pattern as Assignment 22), the new part was wrapping it behind FastAPI with
proper request validation and error handling instead of just calling
functions directly in a notebook.
